# KonkaniVani ASR - Fine-tuning from Checkpoint

## 🔄 Fine-tuning Mode Enabled

This notebook supports both:
- **Training from scratch** (if no checkpoint found)
- **Fine-tuning** (if checkpoint is provided)

## What's Different from Original:
- ✅ Automatic checkpoint detection and loading
- ✅ Reduced learning rate for fine-tuning (0.00001 vs 0.0001)
- ✅ Fewer epochs for fine-tuning (50 vs 100)
- ✅ Preserves pre-trained weights
- ✅ Multi-GPU support

## How to Use:
1. **For fine-tuning**: Add your checkpoint dataset as input
2. **For training from scratch**: Just run without checkpoint
3. Update the `CHECKPOINT_PATH` in Step 1.5 if needed

## Step 1: Setup Environment

In [ ]:
# Install dependencies
!pip install -q torch torchaudio librosa soundfile jiwer pyyaml tensorboard matplotlib

# Suppress dependency warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
import sys
import json
import torch
import torchaudio
from pathlib import Path
import numpy as np
from tqdm import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU count: {torch.cuda.device_count()}")

## Step 1.5: 🔄 Load Pre-trained Checkpoint (Fine-tuning)

In [ ]:
# ============================================================
# CHECKPOINT LOADING FOR FINE-TUNING
# ============================================================

import torch
import os

# UPDATE THIS PATH to your checkpoint dataset!
# Common paths:
# - If uploaded as dataset: '/kaggle/input/your-checkpoint-dataset/best_model.pt'
# - If in konkani-asr-checkpoint: '/kaggle/input/konkani-asr-checkpoint/best_model (1).pt'
CHECKPOINT_PATH = '/kaggle/input/konkani-asr-checkpoint/best_model (1).pt'

# Initialize flags
RESUME_TRAINING = False
PRETRAINED_CHECKPOINT = None
CHECKPOINT_EPOCH = 0
CHECKPOINT_VAL_LOSS = float('inf')

print("="*60)
print("🔍 CHECKING FOR PRE-TRAINED CHECKPOINT")
print("="*60)

if os.path.exists(CHECKPOINT_PATH):
    print(f"\n🔄 Found checkpoint at: {CHECKPOINT_PATH}")
    
    try:
        # Load checkpoint
        print("   Loading checkpoint...")
        checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')
        
        # Display checkpoint info
        print(f"\n📊 Checkpoint Information:")
        print(f"   Epoch: {checkpoint.get('epoch', 'N/A')}")
        print(f"   Validation Loss: {checkpoint.get('val_loss', 'N/A'):.4f}")
        
        # Extract config from checkpoint
        checkpoint_config = checkpoint.get('config', {})
        if checkpoint_config:
            print(f"\n🔧 Original Training Config:")
            print(f"   CTC Weight: {checkpoint_config.get('ctc_weight', 'N/A')}")
            print(f"   Learning Rate: {checkpoint_config.get('learning_rate', 'N/A')}")
        
        # Check what's in the checkpoint
        print(f"\n📦 Checkpoint Contents:")
        for key in checkpoint.keys():
            if key.endswith('_state_dict'):
                print(f"   ✓ {key}")
            else:
                print(f"   - {key}")
        
        # Store for later use
        PRETRAINED_CHECKPOINT = checkpoint
        CHECKPOINT_EPOCH = checkpoint.get('epoch', 0)
        CHECKPOINT_VAL_LOSS = checkpoint.get('val_loss', float('inf'))
        RESUME_TRAINING = True
        
        # Copy to working directory for training script
        working_checkpoint_path = '/kaggle/working/pretrained_checkpoint.pt'
        torch.save(checkpoint, working_checkpoint_path)
        print(f"\n💾 Checkpoint copied to: {working_checkpoint_path}")
        
        print(f"\n{'='*60}")
        print("✅ FINE-TUNING MODE ACTIVATED")
        print(f"{'='*60}")
        print(f"   Will resume from epoch {CHECKPOINT_EPOCH}")
        print(f"   Previous best val loss: {CHECKPOINT_VAL_LOSS:.4f}")
        print(f"   Using reduced learning rate for fine-tuning")
        print(f"{'='*60}\n")
        
    except Exception as e:
        print(f"\n❌ Error loading checkpoint: {e}")
        print("   Will train from scratch instead.")
        RESUME_TRAINING = False
        PRETRAINED_CHECKPOINT = None
        
else:
    print(f"\n⚠️  Checkpoint not found at: {CHECKPOINT_PATH}")
    print("\n   Available input datasets:")
    !ls -lh /kaggle/input/ 2>/dev/null || echo "   (none)"
    print(f"\n{'='*60}")
    print("🆕 TRAINING FROM SCRATCH MODE")
    print(f"{'='*60}")
    print("   No checkpoint provided")
    print("   Will initialize model with random weights")
    print("   Using standard learning rate")
    print(f"{'='*60}\n")

## Step 2: Check Dataset

In [ ]:
# List available datasets
!ls -lh /kaggle/input/

In [ ]:
# Set paths - UPDATE THESE to match your dataset names
from pathlib import Path

DATA_ROOT = Path('/kaggle/input/konkani-asr-complete-data')  # Your main data
SCRIPTS_ROOT = Path('/kaggle/input/kaggle-training-scripts')  # Your scripts dataset

print(f"Data dataset: {DATA_ROOT}")
print(f"Scripts dataset: {SCRIPTS_ROOT}")
print("\nDataset structure:")
!ls -lh {DATA_ROOT}
print("\nScripts structure:")
!ls -lh {SCRIPTS_ROOT}

## Step 3: Extract and Prepare Data

In [ ]:
# Copy training scripts from scripts dataset
import shutil
import zipfile

# Check if datasets exist
if not SCRIPTS_ROOT.exists():
    print(f"✗ ERROR: Scripts dataset not found at {SCRIPTS_ROOT}")
    print("\nPlease add the 'kaggle-training-scripts' dataset as input to this notebook.")
    print("Click 'Add Input' → Search for your scripts dataset → Add it")
    raise FileNotFoundError(f"Scripts dataset not found: {SCRIPTS_ROOT}")

if not DATA_ROOT.exists():
    print(f"✗ ERROR: Data dataset not found at {DATA_ROOT}")
    print("\nPlease add your data dataset as input to this notebook.")
    print("Click 'Add Input' → Search for your data dataset → Add it")
    print("\nAvailable datasets:")
    !ls -la /kaggle/input/
    raise FileNotFoundError(f"Data dataset not found: {DATA_ROOT}")

print("✓ Both datasets found!")
print("\nCopying training scripts...")
script_dirs = ['training_scripts', 'models', 'scripts', 'data']

for dir_name in script_dirs:
    src_dir = SCRIPTS_ROOT / 'tmp' / 'kaggle_scripts_package' / dir_name
    if src_dir.exists():
        dst_dir = Path('/kaggle/working') / dir_name
        shutil.copytree(src_dir, dst_dir, dirs_exist_ok=True)
        print(f"  ✓ Copied {dir_name}/")
    else:
        print(f"  ⚠️  {dir_name}/ not found at {src_dir}")

print("\n✓ Training scripts ready!")

# Extract data files if zipped
zip_files = list(DATA_ROOT.glob('*.zip'))
if zip_files:
    print(f"\nFound {len(zip_files)} data zip files. Extracting...")
    for zip_file in zip_files:
        print(f"Extracting {zip_file.name}...")
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall('/kaggle/working/')
    print("✓ Data extraction complete!")
else:
    print("\nNo data zip files found, copying data directly...")
    # Copy data files directly if not zipped
    for item in DATA_ROOT.iterdir():
        if item.is_dir():
            dst = Path('/kaggle/working') / item.name
            if not dst.exists():
                shutil.copytree(item, dst)
                print(f"  ✓ Copied {item.name}/")

In [ ]:
# Verify extracted files
print("Checking extracted structure...")
print("\nPython scripts:")
!ls -lh /kaggle/working/training_scripts/*.py 2>/dev/null || echo "  ✗ training_scripts not found"
!ls -lh /kaggle/working/models/*.py 2>/dev/null || echo "  ✗ models not found"
!ls -lh /kaggle/working/scripts/*.py 2>/dev/null || echo "  ✗ scripts not found"

print("\nData files:")
!ls -lh /kaggle/working/ | head -15

## Step 4: Locate Dataset Manifests

In [ ]:
# Find manifest files in the extracted data
import os
import json

# Search for manifest directories
possible_manifest_dirs = [
    '/kaggle/working/konkani-10k',  # Your dataset location
    '/kaggle/working/data/konkani-combined/manifests',
    '/kaggle/working/data/konkani-asr-v0/splits/manifests',
    '/kaggle/working/data/manifests'
]

manifest_dir = None
for dir_path in possible_manifest_dirs:
    # Check if directory exists and has manifest files
    if os.path.exists(dir_path):
        test_files = ['train_manifest.json', 'val_manifest.json', 'test_manifest.json']
        if any(os.path.exists(os.path.join(dir_path, f)) for f in test_files):
            manifest_dir = Path(dir_path)
            print(f"✓ Found manifests at: {manifest_dir}")
            break

if manifest_dir is None:
    print("✗ No manifest directory found!")
    print("\nSearching for manifest files...")
    !find /kaggle/working -name "*manifest*.json" 2>/dev/null | grep -v "\._" | head -10

In [ ]:
# Verify manifests and show dataset statistics
if manifest_dir and manifest_dir.exists():
    print("✓ Dataset manifests found:")
    print("="*60)
    
    total_duration = 0
    # Try both naming conventions
    manifest_files = [
        ('train_manifest.json', 'train.json'),
        ('val_manifest.json', 'val.json'),
        ('test_manifest.json', 'test.json')
    ]
    
    for primary_name, alt_name in manifest_files:
        manifest_path = manifest_dir / primary_name
        if not manifest_path.exists():
            manifest_path = manifest_dir / alt_name
        
        if manifest_path.exists():
            # Try loading as JSONL (one JSON per line) or JSON array
            data = []
            with open(manifest_path) as f:
                content = f.read().strip()
                try:
                    # Try as JSON array first
                    data = json.loads(content)
                except json.JSONDecodeError:
                    # Try as JSONL (one JSON object per line)
                    for line in content.split('\n'):
                        if line.strip():
                            data.append(json.loads(line))
            
            num_samples = len(data)
            
            # Calculate total duration if available
            duration = sum(item.get('duration', 0) for item in data)
            total_duration += duration
            
            display_name = manifest_path.name
            print(f"  {display_name:20s}: {num_samples:5,d} samples ({duration/3600:.1f}h)")
        else:
            print(f"  {primary_name:20s}: NOT FOUND")
    
    print("="*60)
    print(f"  Total Duration: {total_duration/3600:.1f} hours")
    print("✓ Ready to train!")
else:
    print("✗ ERROR: No manifest files found!")
    print("Cannot proceed with training.")

## Step 5: Configure Training (Auto-adjusts for Fine-tuning)

In [ ]:
# Training configuration with AUTO-ADJUSTMENT for fine-tuning
import yaml

# Determine learning rate and epochs based on mode
if RESUME_TRAINING:
    # Fine-tuning: Use 10x smaller learning rate and fewer epochs
    learning_rate = 0.00001
    num_epochs = 50
    mode_name = "FINE-TUNING"
    print("🔄 FINE-TUNING MODE DETECTED")
    print(f"  Using reduced learning rate: {learning_rate}")
    print(f"  Using fewer epochs: {num_epochs}")
else:
    # Training from scratch: Use normal learning rate
    learning_rate = 0.0001
    num_epochs = 100
    mode_name = "FROM SCRATCH"
    print("🆕 TRAINING FROM SCRATCH MODE")
    print(f"  Using standard learning rate: {learning_rate}")
    print(f"  Using full epochs: {num_epochs}")

config = {
    'model': {
        'vocab_size': 82,  # Will be updated after vocab generation
        'input_dim': 80,
        'd_model': 128,
        'encoder_layers': 8,
        'decoder_layers': 6,
        'num_heads': 4,
        'conv_kernel_size': 31,
        'dropout': 0.3
    },
    'training': {
        'learning_rate': learning_rate,
        'weight_decay': 0.0001,
        'grad_clip': 5.0,
        'ctc_weight': 0.9,
        'batch_size': 8,
        'gradient_accumulation_steps': 2,
        'mixed_precision': True,
        'num_epochs': num_epochs,
        'save_every': 5,
        'test_every': 5
    },
    'data': {
        'train_manifest': str(manifest_dir / 'train_manifest.json') if (manifest_dir / 'train_manifest.json').exists() else str(manifest_dir / 'train.json'),
        'val_manifest': str(manifest_dir / 'val_manifest.json') if (manifest_dir / 'val_manifest.json').exists() else str(manifest_dir / 'val.json'),
        'vocab_file': str(manifest_dir / 'vocab.json') if (manifest_dir / 'vocab.json').exists() else '/kaggle/working/konkani-10k/vocab.json',
        'num_workers': 2
    },
    'paths': {
        'checkpoint_dir': '/kaggle/working/checkpoints',
        'log_dir': '/kaggle/working/logs'
    },
    'device': 'cuda'
}

# Save config
os.makedirs('/kaggle/working/config', exist_ok=True)
config_filename = f'training_config_{mode_name.lower().replace(" ", "_")}.yaml'
with open(f'/kaggle/working/config/{config_filename}', 'w') as f:
    yaml.dump(config, f)

print(f"\n✓ Training config saved: {config_filename}")
print(f"  Mode: {mode_name}")
print(f"  CTC weight: {config['training']['ctc_weight']}")
print(f"  Learning rate: {config['training']['learning_rate']}")
print(f"  Gradient clip: {config['training']['grad_clip']}")
print(f"  Total epochs: {config['training']['num_epochs']}")
print(f"  Testing: Every {config['training']['test_every']} epochs")

In [ ]:
# Fix audio paths in manifests to point to Kaggle locations
print("Fixing audio paths in manifests...")

for manifest_name in ['train_manifest.json', 'val_manifest.json', 'test_manifest.json']:
    manifest_path = manifest_dir / manifest_name
    if not manifest_path.exists():
        manifest_path = manifest_dir / manifest_name.replace('_manifest', '')
    
    if manifest_path.exists():
        # Read manifest
        with open(manifest_path) as f:
            content = f.read().strip()
        
        # Parse as JSONL
        data = []
        for line in content.split('\n'):
            if line.strip():
                try:
                    data.append(json.loads(line))
                except:
                    pass
        
        # Fix paths
        fixed_count = 0
        for item in data:
            if 'audio_filepath' in item:
                old_path = item['audio_filepath']
                # Extract just the filename part after 'konkani-10k/audio/'
                if 'konkani-10k' in old_path or 'KonkaniRawSpeechCorpus' in old_path:
                    # Get the relative path from audio directory
                    if 'audio/' in old_path:
                        rel_path = old_path.split('audio/')[-1]
                    elif 'Data/' in old_path:
                        rel_path = old_path.split('Data/')[-1]
                    else:
                        rel_path = old_path.split('/')[-1]
                    
                    # Set new path
                    item['audio_filepath'] = f'/kaggle/working/konkani-10k/audio/Data/{rel_path}'
                    fixed_count += 1
        
        # Save fixed manifest
        with open(manifest_path, 'w') as f:
            for item in data:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        
        print(f"  ✓ {manifest_path.name}: Fixed {fixed_count}/{len(data)} paths")

print("\n✓ Manifest paths fixed!")

## Step 6: Setup Python Path and Generate Vocabulary

In [ ]:
# Enable multi-GPU training with DataParallel
import torch

# Check GPU count
num_gpus = torch.cuda.device_count()
print(f"Using {num_gpus} GPU(s) for training")

if num_gpus > 1:
    print("✓ Multi-GPU training enabled!")
    print(f"  Batch size per GPU: {config['training']['batch_size']}")
    print(f"  Total batch per step: {config['training']['batch_size'] * num_gpus}")
    print(f"  Effective batch: {config['training']['batch_size'] * num_gpus * config['training']['gradient_accumulation_steps']}")

In [ ]:
# Add working directory to Python path so imports work
import sys
sys.path.insert(0, '/kaggle/working')

print("Python path configured:")
print(f"  Working dir: /kaggle/working")
print(f"\nVerifying imports...")

try:
    from models.konkanivani_asr import create_konkanivani_model
    print("  ✓ models.konkanivani_asr")
except ImportError as e:
    print(f"  ✗ models.konkanivani_asr: {e}")

try:
    from data.audio_processing.audio_processor import AudioProcessor
    print("  ✓ data.audio_processing.audio_processor")
except ImportError as e:
    print(f"  ✗ data.audio_processing.audio_processor: {e}")

print("\n✓ Ready to train!")

In [ ]:
# Generate Custom Vocabulary from Training Data
print("🔧 Generating custom vocabulary from training data...")

import json
from collections import Counter
import os

def generate_custom_vocab_from_manifests(manifest_paths, min_freq=2):
    """Generate vocabulary from actual training data"""
    
    print("📊 Analyzing training data to build custom vocabulary...")
    
    # Collect all characters from training texts
    char_counter = Counter()
    total_samples = 0
    
    for manifest_path in manifest_paths:
        if not os.path.exists(manifest_path):
            print(f"⚠️  Manifest not found: {manifest_path}")
            continue
            
        print(f"  Processing: {os.path.basename(manifest_path)}")
        
        with open(manifest_path, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f, 1):
                try:
                    data = json.loads(line.strip())
                    text = data.get('text', '')
                    
                    # Count characters in this text
                    for char in text:
                        char_counter[char] += 1
                    
                    total_samples += 1
                    
                    if line_num % 1000 == 0:
                        print(f"    Processed {line_num} samples...")
                        
                except json.JSONDecodeError:
                    continue
    
    print(f"\n📈 Analysis complete:")
    print(f"  Total samples: {total_samples:,}")
    print(f"  Unique characters found: {len(char_counter):,}")
    
    # Filter characters by frequency
    filtered_chars = [char for char, freq in char_counter.items() if freq >= min_freq]
    print(f"  Characters with freq >= {min_freq}: {len(filtered_chars)}")
    
    # Essential tokens (always include)
    essential_tokens = ['<pad>', '<blank>', '<sos>', '<eos>', '<unk>']
    
    # Build vocabulary
    vocab_chars = essential_tokens + sorted(filtered_chars)
    
    # Remove duplicates while preserving order
    seen = set()
    unique_vocab = []
    for char in vocab_chars:
        if char not in seen:
            unique_vocab.append(char)
            seen.add(char)
    
    # Create char2idx and idx2char mappings
    char2idx = {char: idx for idx, char in enumerate(unique_vocab)}
    idx2char = {idx: char for idx, char in enumerate(unique_vocab)}
    
    # Show character frequency stats
    print(f"\n📊 Character frequency analysis:")
    most_common = char_counter.most_common(20)
    for char, freq in most_common:
        display_char = repr(char) if char in [' ', '\n', '\t'] else char
        print(f"  {display_char:>8}: {freq:,}")
    
    return {
        'char2idx': char2idx,
        'idx2char': idx2char,
        'vocab_size': len(char2idx),
        'char_frequencies': dict(char_counter),
        'total_samples': total_samples
    }

# Generate custom vocabulary
manifest_files = []
for manifest_name in ['train_manifest.json', 'val_manifest.json', 'train.json', 'val.json']:
    manifest_path = manifest_dir / manifest_name
    if manifest_path.exists():
        manifest_files.append(str(manifest_path))

if manifest_files:
    print(f"Found {len(manifest_files)} manifest files:")
    for f in manifest_files:
        print(f"  - {f}")
    
    # Generate vocabulary with minimum frequency of 2
    custom_vocab = generate_custom_vocab_from_manifests(manifest_files, min_freq=2)
    
    # Save custom vocabulary
    custom_vocab_path = '/kaggle/working/custom_vocab.json'
    with open(custom_vocab_path, 'w', encoding='utf-8') as f:
        json.dump({
            'char2idx': custom_vocab['char2idx'],
            'idx2char': custom_vocab['idx2char'],
            'vocab_size': custom_vocab['vocab_size']
        }, f, ensure_ascii=False, indent=2)
    
    print(f"\n✅ Custom vocabulary generated!")
    print(f"  Vocabulary size: {custom_vocab['vocab_size']} characters")
    print(f"  Saved to: {custom_vocab_path}")
    print(f"  Based on {custom_vocab['total_samples']:,} training samples")
    
    # Show sample of vocabulary
    print(f"\n📝 Sample vocabulary (first 20 characters):")
    for i, char in enumerate(list(custom_vocab['char2idx'].keys())[:20]):
        display_char = repr(char) if char in [' ', '\n', '\t'] else char
        print(f"  {i:2d}: {display_char}")
    
    # Update config to use custom vocabulary
    config['data']['vocab_file'] = custom_vocab_path
    config['model']['vocab_size'] = custom_vocab['vocab_size']
    
    print(f"\n🔧 Updated config:")
    print(f"  vocab_file: {config['data']['vocab_file']}")
    print(f"  vocab_size: {config['model']['vocab_size']}")
    
else:
    print("❌ No manifest files found! Cannot generate custom vocabulary.")
    print("Using fallback vocabulary...")

## Step 7: Start Training (Auto-loads checkpoint if fine-tuning)

In [ ]:
# ============================================================
# START TRAINING WITH AUTOMATIC CHECKPOINT LOADING
# ============================================================

import sys
sys.path.insert(0, '/kaggle/working')

# Import required modules
from models.konkanivani_asr import create_konkanivani_model
from data.audio_processing.dataset import create_dataloaders
from data.audio_processing.text_tokenizer import KonkaniTokenizer
from training_scripts.train_konkanivani_asr import ASRTrainer
import torch.nn as nn

print("="*60)
if RESUME_TRAINING:
    print("🔄 FINE-TUNING MODE")
    print(f"   Resuming from checkpoint epoch {CHECKPOINT_EPOCH}")
    print(f"   Previous best val loss: {CHECKPOINT_VAL_LOSS:.4f}")
else:
    print("🆕 TRAINING FROM SCRATCH")
print("="*60)

# Load tokenizer
tokenizer = KonkaniTokenizer(config['data']['vocab_file'])
print(f"\n📚 Vocabulary size: {tokenizer.vocab_size}")

# Update model config with actual vocab size
config['model']['vocab_size'] = tokenizer.vocab_size

# Create dataloaders
print(f"\n📊 Creating dataloaders...")
train_loader, val_loader = create_dataloaders(
    config['data']['train_manifest'],
    config['data']['val_manifest'],
    tokenizer,
    batch_size=config['training']['batch_size'],
    num_workers=config['data']['num_workers']
)
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")

# Create model
print(f"\n🏗️  Creating model...")
model = create_konkanivani_model(
    vocab_size=tokenizer.vocab_size, 
    config=config['model']
)

# Count parameters
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Model parameters: {num_params:,}")

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_gpus = torch.cuda.device_count()

print(f"\n🖥️  Device setup:")
print(f"  Device: {device}")
print(f"  GPUs available: {num_gpus}")

# 🔥 LOAD PRE-TRAINED WEIGHTS IF FINE-TUNING
if RESUME_TRAINING and PRETRAINED_CHECKPOINT is not None:
    print(f"\n🔄 Loading pre-trained model weights...")
    try:
        # Load model state dict
        model.load_state_dict(PRETRAINED_CHECKPOINT['model_state_dict'])
        print("  ✅ Model weights loaded successfully!")
        
        print(f"  📊 Resuming from:")
        print(f"     - Epoch: {CHECKPOINT_EPOCH}")
        print(f"     - Best val loss: {CHECKPOINT_VAL_LOSS:.4f}")
        print(f"  🎯 Fine-tuning with:")
        print(f"     - Learning rate: {config['training']['learning_rate']} (10x smaller)")
        print(f"     - Epochs: {config['training']['num_epochs']} (fewer epochs)")
        
    except Exception as e:
        print(f"  ❌ Error loading weights: {e}")
        print("  ⚠️  Will train from scratch instead")
        RESUME_TRAINING = False

# Multi-GPU setup
if num_gpus > 1:
    print(f"\n🚀 Enabling multi-GPU training ({num_gpus} GPUs)")
    model = nn.DataParallel(model)
    effective_batch_size = config['training']['batch_size'] * num_gpus * config['training']['gradient_accumulation_steps']
    print(f"  Batch size per GPU: {config['training']['batch_size']}")
    print(f"  Total batch per step: {config['training']['batch_size'] * num_gpus}")
    print(f"  Effective batch size: {effective_batch_size}")

# Create trainer
print(f"\n🎯 Creating trainer...")
trainer = ASRTrainer(
    model=model,
    tokenizer=tokenizer,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    config=config['training']
)

# Set best val loss from checkpoint if fine-tuning
if RESUME_TRAINING and PRETRAINED_CHECKPOINT is not None:
    trainer.best_val_loss = CHECKPOINT_VAL_LOSS
    print(f"  ✓ Set best_val_loss to {CHECKPOINT_VAL_LOSS:.4f} from checkpoint")
    print("  🔄 Using fresh optimizer for fine-tuning (recommended)")

# Start training
print(f"\n{'='*60}")
print(f"🚀 STARTING TRAINING")
print(f"{'='*60}")
print(f"  Mode: {'Fine-tuning' if RESUME_TRAINING else 'From scratch'}")
print(f"  Learning rate: {config['training']['learning_rate']}")
print(f"  Epochs: {config['training']['num_epochs']}")
print(f"  CTC weight: {config['training']['ctc_weight']}")
print(f"  Checkpoints: {config['paths']['checkpoint_dir']}")
print(f"{'='*60}\n")

# Train!
trainer.train(num_epochs=config['training']['num_epochs'])

print(f"\n{'='*60}")
print(f"✅ TRAINING COMPLETE!")
print(f"{'='*60}")
print(f"  Best model saved to: {config['paths']['checkpoint_dir']}/best_model.pt")
print(f"  Logs saved to: {config['paths']['log_dir']}")
print(f"\nDownload your trained model:")

## Step 8: Download Results

In [ ]:
# Download the best model
from IPython.display import FileLink

print("📥 Download your fine-tuned model:")
FileLink('/kaggle/working/checkpoints/best_model.pt')

In [ ]:
# List all checkpoints
print("\n📁 All checkpoints:")
!ls -lh /kaggle/working/checkpoints/

## 🎉 Training Complete!

### What You Got:
- ✅ Fine-tuned ASR model (or trained from scratch)
- ✅ Checkpoints saved every 5 epochs
- ✅ Best model based on validation loss
- ✅ TensorBoard logs for visualization

### Next Steps:
1. **Download** the best model using the link above
2. **Test** the model on your test set
3. **Compare** with the original checkpoint (if fine-tuning)
4. **Iterate** if needed with more data or different hyperparameters

### Fine-tuning Tips:
- If validation loss increases: Learning rate might be too high
- If no improvement: Model might have already converged
- If overfitting: Increase dropout or add more data